# IntentIQ - Week 2 API Notebook
**CoreSmart GenAI Developer Course · Week 2**

IntentIQ classifies a user message into one of four intents (`cancel_subscription`, `request_refund`, `payment_question`, `unknown`) and benchmarks three providers side by side: **OpenAI** (gpt-5.4-mini), **nano** (gpt-5.4-nano-2026-03-17), and **Ollama** (qwen3:0.6b - local). This notebook covers every endpoint with cURL and Python examples.

Each endpoint is shown two ways:
- **cURL (Windows cmd)** - `%%cmd` cell magic, Windows double-quote syntax
- **Python** - `requests` library, works everywhere

---
### Before you start
1. Server running: `uvicorn app.main:app --reload`
2. `.env` filled with `OPENAI_API_KEY` (covers both OpenAI providers - see `.env.example`)
3. Ollama running locally: `ollama serve` then `ollama pull qwen3:0.6b`
4. Run the **Setup** cell below once

> **Windows note:** All curl cells use `%%cmd` with `\"` to escape inner quotes. Single quotes are not supported in Windows cmd.

In [10]:
# Setup - run this cell first
import requests, json, textwrap

BASE      = 'http://localhost:8000'

# Primary demo message used in curl cells (ASCII only - no special characters)
DEMO_NOTES = 'can you cancel my plan'

# Week-specific: additional examples covering all four intent labels
DEMO = {
    'cancel':   'can you cancel my plan',
    'refund':   'i need a refund asap',
    'billing':  'what is my current bill',
    'ambiguous':'please cancel and refund me',
    'unknown':  'asdf hello',
}

print('Setup complete.')
print('BASE      :', BASE)

Setup complete.
BASE      : http://localhost:8000


---
## 1 · Health Check - `GET /health`
Confirms the server is alive and shows which model is loaded.

In [11]:
%%cmd
curl -s http://localhost:8000/health

Microsoft Windows [Version 10.0.26200.8524]
(c) Microsoft Corporation. All rights reserved.

week 2/>curl -s http://localhost:8000/health
{"status":"ok","model":"gpt-5.4-mini-2026-03-17","openai_model":"gpt-5.4-mini-2026-03-17","nano_model":"gpt-5.4-nano-2026-03-17","ollama_model":"qwen3:0.6b","ollama_base_url":"http://localhost:11434/v1","providers_configured":["openai","nano","ollama"]}
week 2/>

In [12]:
# Health check - Python
r = requests.get(f'{BASE}/health')
print('Status :', r.status_code)
print(json.dumps(r.json(), indent=2))

Status : 200
{
  "status": "ok",
  "model": "gpt-5.4-mini-2026-03-17",
  "openai_model": "gpt-5.4-mini-2026-03-17",
  "nano_model": "gpt-5.4-nano-2026-03-17",
  "ollama_model": "qwen3:0.6b",
  "ollama_base_url": "http://localhost:11434/v1",
  "providers_configured": [
    "openai",
    "nano",
    "ollama"
  ]
}


---
## 2 · Single Classification - `POST /classify`

Classifies one message via a chosen provider and returns a unified `Result` object.

Key concept: the **provider abstraction layer** - every adapter normalises its response into the same `Result` shape regardless of how the underlying API formats its output.

Request body:
```json
{ "text": "<user message>", "provider": "openai" | "nano" | "ollama" }
```

Response (`Result` schema):
```json
{
  "provider": "openai",
  "label": "cancel_subscription",
  "confidence": 0.97,
  "latency_ms": 542.3,
  "input_tokens": 68,
  "output_tokens": 12,
  "raw": { ... }
}
```

> Requires `OPENAI_API_KEY` for `openai` and `nano`. `ollama` needs no key - just `ollama serve` running locally.

In [13]:
%%cmd
curl -s -X POST http://localhost:8000/classify -H "Content-Type: application/json" -d "{\"text\": \"can you cancel my plan\", \"provider\": \"openai\"}"

Microsoft Windows [Version 10.0.26200.8524]
(c) Microsoft Corporation. All rights reserved.

week 2/>curl -s -X POST http://localhost:8000/classify -H "Content-Type: application/json" -d "{\"text\": \"can you cancel my plan\", \"provider\": \"openai\"}"
{"provider":"openai","label":"cancel_subscription","confidence":0.98,"latency_ms":3430.0200999932713,"input_tokens":62,"output_tokens":18,"raw":{"content":"{\"intent\":\"cancel_subscription\",\"confidence\":0.98}"}}
week 2/>

In [14]:
# Single classify - Python (OpenAI)
r = requests.post(f'{BASE}/classify', json={'text': DEMO_NOTES, 'provider': 'openai'})
if r.status_code != 200:
    print(f'Error {r.status_code}:', r.json())
else:
    data = r.json()
    print('Status :', r.status_code)
    print('\n-- Result --')
    print('provider   :', data['provider'])
    print('label      :', data['label'])
    print('confidence :', data['confidence'])
    print('latency_ms :', data['latency_ms'])
    print('tokens     :', data['input_tokens'], 'in /', data['output_tokens'], 'out')

Status : 200

-- Result --
provider   : openai
label      : cancel_subscription
confidence : 0.98
latency_ms : 1044.2731000002823
tokens     : 62 in / 18 out


### 2b · Same message - all three providers side by side

This demonstrates **schema drift**: the same prompt produces three different response shapes on the wire. The adapter layer normalises all of them into the same `Result` object above the abstraction.

In [15]:
# All providers - same message, side by side
message = DEMO_NOTES
print(f'Message: "{message}"\n')
print(f'{"provider":<12} {"label":<26} {"conf":<6} lat')
print('-' * 53)
for provider in ['openai', 'nano', 'ollama']:
    r = requests.post(f'{BASE}/classify', json={'text': message, 'provider': provider})
    if r.status_code != 200:
        print(f'{provider:<12} ERROR: {r.json().get("detail", "?")}')
    else:
        data = r.json()
        print(f'{data["provider"]:<12} {data["label"]:<26} {data["confidence"]:.3f}  {data["latency_ms"]:.0f}ms')

Message: "can you cancel my plan"

provider     label                      conf   lat
-----------------------------------------------------
openai       cancel_subscription        0.960  1418ms
nano         cancel_subscription        0.930  1112ms
ollama       cancel_subscription        1.000  8169ms


---
## 3 · Full Raw Response

Shows the complete JSON as returned by the API - useful for inspecting the `raw` field which contains the original provider response before normalisation.

In [16]:
# Full raw response dump
r = requests.post(f'{BASE}/classify', json={'text': DEMO_NOTES, 'provider': 'openai'})
print(json.dumps(r.json(), indent=2))

{
  "provider": "openai",
  "label": "cancel_subscription",
  "confidence": 0.98,
  "latency_ms": 1281.0869000022649,
  "input_tokens": 62,
  "output_tokens": 18,
  "raw": {
    "content": "{\"intent\":\"cancel_subscription\",\"confidence\":0.98}"
  }
}


---
## 4 · Full Benchmark - `POST /benchmark`

Runs all 30 golden-dataset examples through every configured provider and returns per-provider accuracy, latency percentiles, and cost.

Key concept: the **golden dataset** - 30 hand-labelled examples that give you ground truth to score each provider against.

Response - list of `BenchmarkSummary` objects:
```json
[
  { "provider": "openai", "n": 30, "accuracy": 0.9, "p50_ms": 960,
    "p95_ms": 4665, "cost_per_1k_usd": 0.1293, "cold_start_ms": null },
  ...
]
```

> This is ~90 API calls (30 examples x 3 providers). Expect 30-90 seconds.

In [17]:
%%cmd
curl -s -X POST http://localhost:8000/benchmark

Microsoft Windows [Version 10.0.26200.8524]
(c) Microsoft Corporation. All rights reserved.

week 2/>curl -s -X POST http://localhost:8000/benchmark
[{"provider":"openai","n":30,"accuracy":0.9,"p50_ms":1283.2479499957117,"p95_ms":2254.9071400033426,"cost_per_1k_usd":0.12925,"cold_start_ms":null},{"provider":"nano","n":30,"accuracy":0.8666666666666667,"p50_ms":1315.5848499991407,"p95_ms":3051.951065000685,"cost_per_1k_usd":0.035324999999999995,"cold_start_ms":null},{"provider":"ollama","n":30,"accuracy":0.6666666666666666,"p50_ms":5461.551199998212,"p95_ms":10341.482544997416,"cost_per_1k_usd":0.0,"cold_start_ms":null}]
week 2/>

In [18]:
# Full benchmark - Python
print('Running benchmark (30 examples x 3 providers)...')
r = requests.post(f'{BASE}/benchmark', timeout=300)
if r.status_code != 200:
    print(f'Error {r.status_code}:', r.json())
else:
    summaries = r.json()
    print('Status :', r.status_code)
    print(f"\n{'provider':<11}{'n':>4}{'acc':>8}{'p50':>10}{'p95':>10}{'$/1K':>10}")
    print('-' * 53)
    for s in summaries:
        print(
            f"{s['provider']:<11}{s['n']:>4}{s['accuracy']*100:>7.1f}%"
            f"{s['p50_ms']:>9.0f}ms{s['p95_ms']:>9.0f}ms{s['cost_per_1k_usd']:>9.4f}"
        )

Running benchmark (30 examples x 3 providers)...
Status : 200

provider      n     acc       p50       p95      $/1K
-----------------------------------------------------
openai       30   93.3%     1447ms     4425ms   0.1294
nano         30   93.3%     1523ms     3354ms   0.0354
ollama       30   73.3%     5552ms     8812ms   0.0000


---
## 5 · Streaming Benchmark - `GET /benchmark-stream`

Same benchmark, but streams progress as each example completes via Server-Sent Events.

Key concept: each SSE frame carries a structured event (`start`, `progress`, `provider_done`, `done`) rather than a token delta. The reading boilerplate is identical to the Week 1 streaming pattern.

> **Deviation from standard SSE pattern:** events use `{"type": "..."}` rather than `{"delta": "..."}` - the payload shape reflects the benchmark's structured output rather than token-by-token text.

In [19]:
%%cmd
curl -s -N http://localhost:8000/benchmark-stream

Microsoft Windows [Version 10.0.26200.8524]
(c) Microsoft Corporation. All rights reserved.

week 2/>curl -s -N http://localhost:8000/benchmark-stream
data: {"type": "start", "providers": ["openai", "nano", "ollama"], "n": 30}

data: {"type": "provider_start", "provider": "openai"}

data: {"type": "progress", "provider": "openai", "done": 1, "total": 30, "correct": 1, "accuracy": 1.0, "avg_lat_ms": 1411.1144999988028}

data: {"type": "progress", "provider": "openai", "done": 2, "total": 30, "correct": 2, "accuracy": 1.0, "avg_lat_ms": 1361.1527999964892}

data: {"type": "progress", "provider": "openai", "done": 3, "total": 30, "correct": 3, "accuracy": 1.0, "avg_lat_ms": 1374.7958999980863}

data: {"type": "progress", "provider": "openai", "done": 4, "total": 30, "correct": 4, "accuracy": 1.0, "avg_lat_ms": 1358.5338999982923}

data: {"type": "progress", "provider": "openai", "done": 5, "total": 30, "correct": 5, "accuracy": 1.0, "avg_lat_ms": 1286.5570800000569}

data: {"type": "progr

In [20]:
# Streaming benchmark - Python (reads SSE events one by one)
# Note: event payload uses {"type": ...} not {"delta": ...}
print('-- Streaming benchmark progress --\n')
with requests.get(f'{BASE}/benchmark-stream', stream=True) as resp:
    resp.raise_for_status()
    for raw_line in resp.iter_lines():
        if not raw_line:
            continue
        line = raw_line.decode() if isinstance(raw_line, bytes) else raw_line
        if not line.startswith('data: '):
            continue
        payload_str = line[6:]
        if payload_str == '[DONE]':
            print('\n-- Stream complete --')
            break
        try:
            event = json.loads(payload_str)
            t = event.get('type', '')
            if t == 'start':
                print(f"Starting: {event['providers']} x {event['n']} examples")
            elif t == 'provider_start':
                print(f"  {event['provider']}: ", end='', flush=True)
            elif t == 'progress':
                if event['done'] % 10 == 0:
                    print(f"{event['done']}/{event['total']} ", end='', flush=True)
            elif t == 'provider_done':
                s = event.get('summary', {})
                if s:
                    print(f"done  acc={s['accuracy']*100:.1f}%  p50={s['p50_ms']:.0f}ms")
            elif t == 'done':
                print('\nAll done.')
        except json.JSONDecodeError:
            pass

-- Streaming benchmark progress --

Starting: ['openai', 'nano', 'ollama'] x 30 examples
  openai: 10/30 20/30 30/30 done  acc=86.7%  p50=1324ms
  nano: 10/30 20/30 30/30 done  acc=86.7%  p50=1510ms
  ollama: 10/30 20/30 30/30 done  acc=70.0%  p50=5723ms

All done.

-- Stream complete --


---
## 6 · CLI Benchmark

Same benchmark, no server needed - runs as a Python module and prints the table directly.

```bash
python -m app.main benchmark
```

In [21]:
%%cmd
python -m app.main benchmark

Microsoft Windows [Version 10.0.26200.8524]
(c) Microsoft Corporation. All rights reserved.

week 2/>python -m app.main benchmark


2026-06-04 22:48:46,390 INFO intentiq - Loading golden dataset from week 2/golden_dataset.jsonl
2026-06-04 22:48:46,392 INFO intentiq - Running benchmark on 30 examples...
2026-06-04 22:48:48,555 INFO httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-04 22:48:50,457 INFO httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-04 22:48:52,075 INFO httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-04 22:48:53,740 INFO httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-04 22:48:55,302 INFO httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-04 22:48:56,825 INFO httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-04 22:48:58,405 INFO httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-0


provider      n     acc       p50       p95      $/1K
-----------------------------------------------------
openai       30   86.7%     1520ms     2203ms   0.1291
nano         30   90.0%     1621ms     2629ms   0.0354
ollama       30   80.0%     5208ms     9486ms   0.0000

week 2/>

---
## 7 · Failure Mode - Invalid Provider (422)

Pydantic validates the `provider` field against `Literal["openai", "nano", "ollama"]`. An unknown string returns **422** before any API call is made - no tokens spent.

In [22]:
%%cmd
curl -s -X POST http://localhost:8000/classify -H "Content-Type: application/json" -d "{\"text\": \"cancel my plan\", \"provider\": \"gpt4\"}"

Microsoft Windows [Version 10.0.26200.8524]
(c) Microsoft Corporation. All rights reserved.

week 2/>curl -s -X POST http://localhost:8000/classify -H "Content-Type: application/json" -d "{\"text\": \"cancel my plan\", \"provider\": \"gpt4\"}"
{"detail":[{"type":"literal_error","loc":["body","provider"],"msg":"Input should be 'openai', 'nano' or 'ollama'","input":"gpt4","ctx":{"expected":"'openai', 'nano' or 'ollama'"}}]}
week 2/>

In [23]:
# Failure: invalid provider name
r = requests.post(f'{BASE}/classify', json={'text': 'cancel my plan', 'provider': 'gpt4'})
print(f'Status: {r.status_code}  (expected 422 -- Pydantic rejects unknown provider, no API call made)')
print(json.dumps(r.json(), indent=2))

Status: 422  (expected 422 -- Pydantic rejects unknown provider, no API call made)
{
  "detail": [
    {
      "type": "literal_error",
      "loc": [
        "body",
        "provider"
      ],
      "msg": "Input should be 'openai', 'nano' or 'ollama'",
      "input": "gpt4",
      "ctx": {
        "expected": "'openai', 'nano' or 'ollama'"
      }
    }
  ]
}


---
## 8 · Failure Mode - Provider Key Not Configured (400)

When a provider's API key is missing from `.env`, the adapter raises a `RuntimeError` which FastAPI converts to **400**. The harness keeps running for other providers.

In [24]:
%%cmd
curl -s -X POST http://localhost:8000/classify -H "Content-Type: application/json" -d "{\"text\": \"cancel my plan\", \"provider\": \"openai\"}"

Microsoft Windows [Version 10.0.26200.8524]
(c) Microsoft Corporation. All rights reserved.

week 2/>curl -s -X POST http://localhost:8000/classify -H "Content-Type: application/json" -d "{\"text\": \"cancel my plan\", \"provider\": \"openai\"}"
{"provider":"openai","label":"cancel_subscription","confidence":0.98,"latency_ms":1530.4749999995693,"input_tokens":60,"output_tokens":18,"raw":{"content":"{\"intent\":\"cancel_subscription\",\"confidence\":0.98}"}}
week 2/>

In [25]:
# Failure: provider key not in .env
# Remove OPENAI_API_KEY from .env temporarily, then restart uvicorn to trigger this.
r = requests.post(f'{BASE}/classify', json={'text': 'cancel my plan', 'provider': 'openai'})
if r.status_code == 400:
    print(f'Status: {r.status_code}  (expected 400 -- key not configured)')
    print(json.dumps(r.json(), indent=2))
else:
    # key IS configured -- shows a real classify result instead
    data = r.json()
    print(f'Key is configured -- got: label={data["label"]}  conf={data["confidence"]:.3f}')

Key is configured -- got: label=cancel_subscription  conf=0.980


---
## 9 · OpenAPI / Swagger Docs
FastAPI auto-generates interactive docs - try endpoints live in the browser:

In [26]:
from IPython.display import display, HTML
display(HTML('<a href="http://localhost:8000/docs" target="_blank" style="font-size:15px">'
             'Open Swagger UI: http://localhost:8000/docs</a>'))